# Physics-informed (PINN) benchmark tasks

Beyond the convection / reaction / wave problems of `create_pde_task`, `l2co_tasks` ships the PINN benchmark suite of [Jnini et al. (2026)](https://arxiv.org/abs/2604.05230), *Curvature-aware optimization for high-accuracy physics-informed neural networks*. These span the elliptic, parabolic, hyperbolic and stiff regimes that make PINN training hard:

| Factory | Problem | Class |
| --- | --- | --- |
| `create_helmholtz_task` | 2-D / 3-D Helmholtz | elliptic, oscillatory |
| `create_viscous_burgers_task` | (2+1)-D viscous Burgers | parabolic |
| `create_inviscid_burgers_task` | inviscid Burgers (entropy / flux relaxation) | hyperbolic, shocks |
| `create_euler_task` | 1-D Euler / Sod shock tube (HLLC) | hyperbolic, shocks |
| `create_stokes_task` | lid-driven wedge (Moffatt eddies) | elliptic, multiscale |
| `create_pkpd_task` | stiff PK-PD ODE (paclitaxel) | stiff, discontinuous forcing |

Every factory returns a standard [`Task`](api.md#l2co_tasks.Task): a model (an `eqx.Module`), a picklable `loss_fn`, a collocation-point dataset (`.npz`), and a `tag`. All carry `global_min = 0.0` (each loss is a sum of mean-squared-error / penalty terms), so regret is measured against a perfectly satisfied PDE.

Below we build a *tiny* instance of each (small networks, few collocation points) so the notebook runs quickly; the factory defaults reproduce the paper's settings.


In [ ]:
import tempfile
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp

from l2co_tasks import (
    Task,
    create_euler_task,
    create_helmholtz_task,
    create_inviscid_burgers_task,
    create_pkpd_task,
    create_stokes_task,
    create_viscous_burgers_task,
)

tmp = Path(tempfile.mkdtemp())

## A single descent step

Each task plugs straight into the canonical loss contract `task.loss_fn(model, **task.loaded_dataset)`. We define a helper that takes one gradient-descent step and checks the loss is finite and decreases.


In [ ]:
def one_step(task, lr=1e-3):
    """Evaluate the loss and take a single gradient-descent step."""
    data = task.loaded_dataset

    def loss(model):
        return task.loss_fn(model, **data)

    value, grads = eqx.filter_value_and_grad(loss)(task.model)
    params, static = eqx.partition(task.model, eqx.is_inexact_array)
    gparams, _ = eqx.partition(grads, eqx.is_inexact_array)
    new = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, gparams)
    after = loss(eqx.combine(new, static))
    return float(value), float(after)

## Helmholtz (elliptic)

A periodic Fourier-feature embedding makes the network exactly periodic, so the loss is the interior residual alone. Pass `dim=3` (with `a3` and `modes=(1, 2)`) for the 3-D problem.


In [ ]:
helmholtz = create_helmholtz_task(
    a1=1.0,
    a2=4.0,
    k=1.0,
    seed=0,
    dataset_path=str(tmp / "helmholtz"),
    dim=2,
    modes=(1,),
    num_res_points=64,
    hidden_size=16,
    num_layers=4,
)
print("tag:", helmholtz.tag)
print("parameters:", helmholtz.dimensionality)
print("loss before/after step:", one_step(helmholtz))

## Viscous Burgers (parabolic)

Initial- and boundary-condition targets are taken from the closed-form solution `u = 1 / (1 + exp((x + y - t) / (2 nu)))`.


In [ ]:
burgers = create_viscous_burgers_task(
    seed=0,
    dataset_path=str(tmp / "vburgers"),
    num_res_points=64,
    num_ic_points=16,
    num_bc_points=16,
    hidden_size=16,
    num_layers=4,
)
print("tag:", burgers.tag)
print("loss before/after step:", one_step(burgers))

## Inviscid Burgers (hyperbolic, two networks)

The entropy / flux-relaxation formulation trains a solution network and a flux network jointly (bundled in a `MultiNet`), with a Tadmor entropy-inequality penalty selecting the physical weak solution.


In [ ]:
inviscid = create_inviscid_burgers_task(
    seed=0,
    dataset_path=str(tmp / "iburgers"),
    num_res_points=64,
    num_ic_points=16,
    num_bc_points=16,
    hidden_size=16,
    num_layers=4,
)
print("sub-networks:", inviscid.model.names)
print("parameters (both nets):", inviscid.dimensionality)
print("loss before/after step:", one_step(inviscid))

## Euler / Sod shock tube (staged)

The Euler problem is solved in two stages selected by `stage`: a viscous warm-up (`stage='viscous'`) followed by the inviscid HLLC residual (`stage='inviscid'`). In a full run you load the converged viscous-stage weights into the inviscid task before continuing; here we just build both.


In [ ]:
euler_viscous = create_euler_task(
    seed=0,
    dataset_path=str(tmp / "euler"),
    stage="viscous",
    num_res_points=64,
    num_ic_points=16,
    num_bc_points=16,
    hidden_size=16,
    num_layers=4,
)
euler_inviscid = create_euler_task(
    seed=0,
    dataset_path=str(tmp / "euler"),
    stage="inviscid",
    num_res_points=64,
    num_ic_points=16,
    num_bc_points=16,
    hidden_size=16,
    num_layers=4,
)
print("viscous loss before/after:", one_step(euler_viscous))
print("inviscid loss before/after:", one_step(euler_inviscid))

## Stokes wedge (elliptic, multiscale)

A lid-driven triangular wedge whose Moffatt eddies decay rapidly toward the apex. There is no closed-form reference, so the loss is the Stokes residual plus the lid / wall boundary conditions.


In [ ]:
stokes = create_stokes_task(
    seed=0,
    dataset_path=str(tmp / "stokes"),
    num_res_points=128,
    num_lid_points=16,
    num_wall_points=16,
    hidden_size=16,
    num_layers=4,
)
print("tag:", stokes.tag)
print("loss before/after step:", one_step(stokes))

## Stiff PK-PD ODE

A coupled pharmacokinetic / pharmacodynamic model of paclitaxel acting on tumour growth. The plasma concentration `c(t)` is analytic (a sum of fast / slow exponentials triggered at each dose), and the network learns the four transit-compartment states on a non-uniform time grid.


In [ ]:
pkpd = create_pkpd_task(
    seed=0,
    dataset_path=str(tmp / "pkpd"),
    point_allocation=((0.0, 1.9, 60), (1.9, 4.0, 60), (4.0, 17.0, 80)),
    hidden_size=16,
    num_layers=4,
)
print("tag:", pkpd.tag)
print("loss before/after step:", one_step(pkpd))

## Save / load round-trip

Like every `Task`, the PINN tasks serialize to a single `.eqx` file (JSON header + binary leaves) alongside their `.npz` dataset.


In [ ]:
path = Task.save(helmholtz, str(tmp / "helmholtz_task"))
loaded = Task.load(path)
print("hash preserved:", loaded.hash == helmholtz.hash)
data = loaded.loaded_dataset
print(
    "loss preserved:",
    jnp.allclose(
        loaded.loss_fn(loaded.model, **data),
        helmholtz.loss_fn(helmholtz.model, **data),
    ),
)